# Step 1 — Quality Control and Station Filtering

This notebook loads raw daily discharge data from the CEDEX database, applies quality-control
filters, and produces a clean dataset of gauging stations suitable for drought analysis.

## Pipeline
1. Load raw discharge records (~4.68 M rows, 302 stations, 1912–2021).
2. Restrict to the study period **1961–2020**.
3. **Filter 1** — Remove stations with > 30% missing values.
4. **Filter 2** — Remove stations with consecutive NaN gaps > 365 days.
5. **Filter 3** — Remove stations with consecutive zero-flow periods > 120 days.
6. Merge with UTM coordinates and export the final dataset.

## Inputs
| File | Description |
|---|---|
| `data/caudales_raw.csv` | Raw CEDEX discharge records |
| `data/station_locations.csv` | Station coordinates (UTM) |

## Outputs
| File | Description |
|---|---|
| `data/caudales_ebro_filtrado.csv` | Filtered discharge data (before zero-flow filter) |
| `data/caudales_diarios_final.csv` | Final filtered discharge data |
| `data/caudales_diarios_final_con_utm.csv` | Final data with UTM coordinates |

In [ ]:
# Load raw discharge data from CEDEX
import pandas as pd
import numpy as np

raw_df = pd.read_csv("data/caudales_raw.csv", sep=";")

# Parse dd/mm/YYYY
raw_df["fecha"] = pd.to_datetime(raw_df["fecha"], dayfirst=True, errors="coerce")

# Ensure numeric types
raw_df["caudal"] = pd.to_numeric(raw_df["caudal"], errors="coerce")
raw_df["altura"] = pd.to_numeric(raw_df["altura"], errors="coerce")

# Basic check
print(raw_df.shape)
print("Dates:", raw_df["fecha"].min(), "->", raw_df["fecha"].max())
print("Stations:", raw_df["indroea"].nunique())
print("NaT dates:", raw_df["fecha"].isna().sum())

In [ ]:
# Filter to 1961-2020 study period
start = pd.Timestamp("1961-01-01")
end   = pd.Timestamp("2020-12-31")

df = raw_df.loc[(raw_df["fecha"] >= start) & (raw_df["fecha"] <= end),
                ["indroea","fecha","caudal"]].copy()

# If duplicates exist (same station-date), keep the mean
df = (df.groupby(["indroea","fecha"], as_index=False)["caudal"]
        .mean())

print(df.shape, "stations:", df["indroea"].nunique())


In [ ]:
# Reindex to have all dates in the range
all_days = pd.date_range(start, end, freq="D")

def reindex_station(g):
    g = g.set_index("fecha").sort_index()
    g = g.reindex(all_days)
    g["indroea"] = g["indroea"].iloc[0]
    return g.reset_index().rename(columns={"index":"fecha"})

full = (df.sort_values(["indroea","fecha"])
          .groupby("indroea", group_keys=False)
          .apply(reindex_station))

print(full.shape)

In [ ]:
# Compute NaN run statistics per station for initial filtering 
def max_run(mask: np.ndarray) -> int:
    run = maxrun = 0
    for v in mask:
        if v:
            run += 1
            maxrun = max(maxrun, run)
        else:
            run = 0
    return maxrun

qc = []
for st, g in full.groupby("indroea"):
    s = g["caudal"].to_numpy()

    nan_mask = np.isnan(s)
    zero_mask = (s == 0)  # note: NaN==0 evaluates to False - OK

    qc.append({
        "indroea": st,
        "n_days": len(s),
        "pct_nan": float(np.mean(nan_mask)),
        "max_nan_run": int(max_run(nan_mask)),
        "pct_zero": float(np.mean(zero_mask)),
        "max_zero_run": int(max_run(zero_mask)),
        "min_q": float(np.nanmin(s)) if np.any(~nan_mask) else np.nan,
        "max_q": float(np.nanmax(s)) if np.any(~nan_mask) else np.nan,
    })

qc = pd.DataFrame(qc)

qc.sort_values("max_nan_run", ascending=False).head(15)

In [ ]:
# Remove stations with > 30% missing values
TH_PCT_NAN = 0.30

qc_30 = qc[qc["pct_nan"] <= TH_PCT_NAN].copy()
ok_stations_30 = qc_30["indroea"].astype(int).tolist()

print("Threshold pct_nan <=", TH_PCT_NAN)
print("Total stations:", qc.shape[0])
print("Pass filter:", len(ok_stations_30))
print("Fail filter:", qc.shape[0] - len(ok_stations_30))

# Inspect the worst stations still passing (near 30%)
qc_30.sort_values("pct_nan", ascending=False).head(10)[
    ["indroea","pct_nan","max_nan_run","pct_zero","max_zero_run","min_q","max_q"]
]

In [ ]:
# Filter the full daily DataFrame 
full_30 = full[full["indroea"].isin(ok_stations_30)].copy()

print("Rows before:", len(full), " | after:", len(full_30))
print("Stations in full_30:", full_30["indroea"].nunique())

In [ ]:
# Summary of selected stations 
print(qc_30[["pct_nan","max_nan_run","pct_zero","max_zero_run"]].describe(percentiles=[.5,.75,.9,.95,.99]))

# Top NaN streaks among passing stations
qc_30.sort_values("max_nan_run", ascending=False).head(15)[
    ["indroea","pct_nan","max_nan_run","pct_zero","max_zero_run"]
]

In [ ]:
# Second filter: max consecutive NaN run (e.g. 1 year = 365 days)
TH_MAX_RUN = 365   

qc_30_run = qc_30[qc_30["max_nan_run"] <= TH_MAX_RUN].copy()
ok_stations_30_run = qc_30_run["indroea"].astype(int).tolist()

print("Filter pct_nan <= 0.30 and max_nan_run <=", TH_MAX_RUN)
print("Stations passing:", len(ok_stations_30_run))

full_30_run = full[full["indroea"].isin(ok_stations_30_run)].copy()
print("Stations in full_30_run:", full_30_run["indroea"].nunique())
print("Rows:", len(full_30_run))

In [ ]:
qc_30_run[["pct_nan","max_nan_run"]].describe()

In [ ]:
# Save filtered dataset 
full_30_run.to_csv("data/caudales_ebro_filtrado.csv", index=False)

### Zero-flow streak analysis\nThe filtered dataset is now cleaner. Next, we examine zero-flow streaks: 

In [ ]:
import numpy as np
import pandas as pd

df0 = full_30_run.sort_values(["indroea","fecha"]).copy()

def zero_run_stats(g, col="caudal"):
    x = g[col].to_numpy()

    # NaN: (NaN==0) is False, so NaN does not count as zero (correct)
    is0 = (x == 0)

    # % zeros
    pct0 = float(is0.mean())

    # max run + number of runs
    run = 0
    maxrun = 0
    n_runs = 0
    for v in is0:
        if v:
            if run == 0:
                n_runs += 1
            run += 1
            if run > maxrun:
                maxrun = run
        else:
            run = 0

    # dates of the longest run (optional, for inspection)
    max_start = None
    max_end = None
    if maxrun > 0:
        run = 0
        start = None
        best = (None, None)
        best_len = 0
        dates = g["fecha"].to_numpy()
        for i, v in enumerate(is0):
            if v and run == 0:
                start = i
            if v:
                run += 1
                if run > best_len:
                    best_len = run
                    best = (start, i)
            else:
                run = 0
        i0, i1 = best
        max_start = pd.to_datetime(dates[i0])
        max_end   = pd.to_datetime(dates[i1])

    return pd.Series({
        "n_days": len(x),
        "pct_zero": pct0,
        "max_zero_run": int(maxrun),
        "n_zero_runs": int(n_runs),
        "max_zero_start": max_start,
        "max_zero_end": max_end,
    })

zero_qc = (df0.groupby("indroea", group_keys=False)
             .apply(zero_run_stats, col="caudal")
             .reset_index())

zero_qc.sort_values("max_zero_run", ascending=False).head(20)

In [ ]:
print(zero_qc[["pct_zero","max_zero_run","n_zero_runs"]].describe(percentiles=[.5,.75,.9,.95,.99]))

In [ ]:
# Filter by max zero-flow run (e.g. 120 days)
TH_ZERO_RUN = 120

ok_zero = zero_qc[zero_qc["max_zero_run"] <= TH_ZERO_RUN]["indroea"].astype(int)

final_stations = set(ok_stations_30_run) & set(ok_zero)

print("Final stations:", len(final_stations))

df_final = full_30_run[full_30_run["indroea"].isin(final_stations)].copy()
print("Final rows:", len(df_final))

In [ ]:
zero_qc[zero_qc["indroea"].isin(final_stations)][
    ["pct_zero","max_zero_run"]
].describe()

In [ ]:
# Save filtered dataset transforming indroea to integer and renaming columns
df_out = (
    df_final
    .assign(indroea=df_final["indroea"].astype(int))
    .rename(columns={
        "indroea": "station_id",
        "fecha": "date",
        "caudal": "Q"
    })
    [["station_id", "date", "Q"]]
    .sort_values(["station_id", "date"])
)

csv_out = "data/caudales_diarios_final.csv"
df_out.to_csv(csv_out, index=False)

print("Saved:", csv_out)
print("Stations:", df_out["station_id"].nunique())
print("Rows:", len(df_out))
print("Date range:", df_out["date"].min(), "->", df_out["date"].max())


### Merge station coordinates 

In [ ]:
import pandas as pd

# --- 1) Load station coordinates ---
coords_path = "data/station_locations.csv"  
coords = pd.read_csv(coords_path)

# Normalize column names
coords = coords.rename(columns={
    "utm_x": "xutm",
    "utm_y": "yutm",
    "station_id": "station_id"
})

# Ensure compatible station_id type
coords["station_id"] = pd.to_numeric(coords["station_id"], errors="coerce").astype("Int64")

# Keep required columns and drop duplicates
coords = coords[["station_id", "xutm", "yutm"]].drop_duplicates("station_id")

# --- 2) Prepare the final daily DataFrame ---


df_out = (
    df_final.rename(columns={"indroea":"station_id", "fecha":"date", "caudal":"Q"})
            [["station_id","date","Q"]]
            .sort_values(["station_id","date"])
            .copy()
)

df_out["station_id"] = pd.to_numeric(df_out["station_id"], errors="coerce").astype("Int64")
df_out["date"] = pd.to_datetime(df_out["date"])

# --- 3) Merge ---
df_out = df_out.merge(coords, on="station_id", how="left")

# --- 4) Checks ---
missing_xy = df_out["xutm"].isna().mean()
print(f"Filas sin coordenadas: {missing_xy:.2%}")

# stations without coordinates
missing_st = (df_out.loc[df_out["xutm"].isna(), "station_id"]
              .dropna().unique())
print("Stations without coords:", len(missing_st))
print(list(missing_st)[:20])

# Drop rows without coordinates
df_out = df_out[~df_out["xutm"].isna()].copy()

# --- 5) Save ---
csv_out = "data/caudales_diarios_final_con_utm.csv"
df_out.to_csv(csv_out, index=False)
print("Saved:", csv_out, "shape:", df_out.shape)


In [ ]:
# Check station count and IDs in the final dataset
print("Rows in final dataset with coords:", len(df_out))
print("Stations in final dataset with coords:", df_out["station_id"].nunique())
print("Station codes in final dataset with coords:", sorted(df_out["station_id"].unique().tolist()))